# Mô hình 3 - LSTM Sequence-to-Vector t+1 đến t+24

LSTM nhận chuỗi quan sát 72 giờ gần nhất và trả về đồng thời vector 24 giá trị PM2.5
tương lai. Khác với 24 XGBoost độc lập, kiến trúc này chia sẻ biểu diễn giữa các
horizon nên có khả năng học quan hệ của cả quỹ đạo dự báo.

Mô hình dùng MSE làm loss, Adam làm optimizer, theo dõi RMSE và MAE trên Train và
Validation, có Early Stopping, ReduceLROnPlateau, Dropout và L2 regularization.


In [1]:
from pathlib import Path
import json
import random
from typing import Any

import joblib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
})
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Nhận diện project bằng cấu trúc, không phụ thuộc tên thư mục tạm của project.
CURRENT_DIR = Path.cwd().resolve()
SEARCH_DIRS = [CURRENT_DIR, CURRENT_DIR.parent]
SEARCH_DIRS.extend(path for path in CURRENT_DIR.iterdir() if path.is_dir())
PROJECT_CANDIDATES = []
for candidate in SEARCH_DIRS:
    data_file = candidate / "data" / "processed" / "pm25_training_data_enriched.csv"
    notebook_marker = candidate / "model" / "0_multihorizon_data_preparation.ipynb"
    if notebook_marker.exists() and data_file.exists():
        resolved = candidate.resolve()
        if resolved not in PROJECT_CANDIDATES:
            PROJECT_CANDIDATES.append(resolved)

if len(PROJECT_CANDIDATES) == 1:
    PROJECT_ROOT = PROJECT_CANDIDATES[0]
elif not PROJECT_CANDIDATES:
    raise FileNotFoundError(
        "Không tìm thấy project chứa đồng thời model và "
        "data/processed/pm25_training_data_enriched.csv."
    )
else:
    raise RuntimeError(
        "Có nhiều project phù hợp; hãy mở Jupyter tại đúng thư mục gốc cần chạy: "
        + ", ".join(str(path) for path in PROJECT_CANDIDATES)
    )

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = MODEL_DIR / "results"
CANDIDATES_DIR = MODEL_DIR / "candidates"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data_enriched.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = np.arange(1, 25, dtype=int)
TARGET_COLUMNS = [f"target_pm25_t_plus_{h}" for h in HORIZONS]
MAX_HORIZON = int(HORIZONS.max())

POLLUTANT_FEATURES = ["pm25", "pm10", "o3", "no2", "so2", "co"]
WEATHER_FEATURES = [
    "temp", "humidity", "wind_speed", "wind_dir", "precip", "pressure", "cloud_cover"
]
TEMPORAL_FEATURES = ["hour", "day_of_week", "month", "is_weekend", "day_of_year"]
HISTORY_FEATURES = [
    "pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h", "pm25_lag_12h",
    "pm25_lag_24h", "pm25_lag_48h", "pm25_lag_72h", "pm25_lag_96h",
    "pm25_lag_120h", "pm25_lag_144h", "pm25_lag_168h",
    "pm25_roll_6h", "pm25_roll_12h", "pm25_roll_24h", "pm25_roll_72h",
    "pm25_roll_168h", "pm25_std_6h", "pm25_std_12h", "pm25_std_24h",
    "pm25_std_72h", "pm25_std_168h", "pm25_min_24h", "pm25_max_24h",
    "pm25_delta_1h", "pm25_delta_3h", "pm25_delta_24h",
    "pm25_roll_ratio_6h_24h", "pm25_roll_ratio_24h_72h",
    "pm25_same_hour_mean_7d", "pm25_same_hour_median_7d",
    "pm25_same_hour_std_7d", "pm25_same_hour_min_7d",
    "pm25_same_hour_max_7d", "pm25_same_hour_ratio_7d", "pm25_weekly_delta",
]
ENGINEERED_FEATURES = [
    "ventilation_index", "humid_stagnation", "rain_flag", "calm_wind",
    "high_humidity", "wind_x", "wind_y", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "month_sin", "month_cos", "pm25_pm10_ratio",
    "no2_co_ratio",
]
NUMERIC_FEATURES = (
    POLLUTANT_FEATURES + WEATHER_FEATURES + TEMPORAL_FEATURES
    + HISTORY_FEATURES + ENGINEERED_FEATURES
)
CATEGORICAL_FEATURES = ["city", "season"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data: {DATA_PATH}")


Project root: D:\Project123456\aqi-vietnam\aqi-vietnam4
Data: D:\Project123456\aqi-vietnam\aqi-vietnam4\data\processed\pm25_training_data_enriched.csv


In [2]:
def make_multihorizon_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Ghép chính xác PM2.5 tại t+1,...,t+24 theo city và timestamp."""
    frame = data.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame = frame.sort_values(["city", "datetime"]).reset_index(drop=True)
    if frame.duplicated(["city", "datetime"]).any():
        raise ValueError("Dữ liệu có city/datetime trùng, không thể ghép target chính xác.")

    lookup = frame.set_index(["city", "datetime"])["pm25"]
    for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
        keys = pd.MultiIndex.from_arrays(
            [frame["city"], frame["datetime"] + pd.to_timedelta(horizon, unit="h")],
            names=["city", "datetime"],
        )
        frame[column] = lookup.reindex(keys).to_numpy(dtype=float)
    frame["forecast_end"] = frame["datetime"] + pd.Timedelta(hours=MAX_HORIZON)
    return frame


def make_temporal_split(
    frame: pd.DataFrame,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    purge_hours: int = MAX_HORIZON,
) -> dict[str, Any]:
    """Chia theo forecast_end để không có cửa sổ target giao nhau giữa các tập."""
    unique_ends = pd.Series(frame["forecast_end"].dropna().sort_values().unique())
    train_cut = pd.Timestamp(unique_ends.iloc[int(len(unique_ends) * train_fraction)])
    val_cut = pd.Timestamp(
        unique_ends.iloc[int(len(unique_ends) * (train_fraction + validation_fraction))]
    )
    purge = pd.Timedelta(hours=purge_hours)
    forecast_end = pd.to_datetime(frame["forecast_end"])
    masks = {
        "train": forecast_end < train_cut,
        "validation": (forecast_end >= train_cut + purge) & (forecast_end < val_cut),
        "test": forecast_end >= val_cut + purge,
    }
    if any(not mask.any() for mask in masks.values()):
        raise ValueError("Temporal split tạo ra ít nhất một tập rỗng.")
    return {
        "masks": masks,
        "train_cut": train_cut,
        "val_cut": val_cut,
        "purge_hours": purge_hours,
    }


def split_summary(frame: pd.DataFrame, split: dict[str, Any]) -> dict[str, Any]:
    summary: dict[str, Any] = {
        "strategy": "global chronological 70/15/15 by forecast_end with 24-hour purge gaps",
        "train_cut": split["train_cut"].isoformat(),
        "validation_cut": split["val_cut"].isoformat(),
        "purge_hours": int(split["purge_hours"]),
        "horizons": HORIZONS.tolist(),
    }
    for name, mask in split["masks"].items():
        part = frame.loc[mask]
        summary[name] = {
            "rows": int(len(part)),
            "source_start": part["datetime"].min().isoformat(),
            "source_end": part["datetime"].max().isoformat(),
            "forecast_end_start": part["forecast_end"].min().isoformat(),
            "forecast_end_end": part["forecast_end"].max().isoformat(),
        }
    return summary


def load_model_frame() -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    raw = pd.read_csv(DATA_PATH, low_memory=False)
    raw["datetime"] = pd.to_datetime(raw["datetime"])
    supervised = make_multihorizon_frame(raw)
    required = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_COLUMNS
    missing_columns = sorted(set(required) - set(supervised.columns))
    if missing_columns:
        raise KeyError(f"Thiếu cột cần thiết: {missing_columns}")
    frame = supervised.dropna(subset=required).copy().reset_index(drop=True)
    split = make_temporal_split(frame)
    manifest = split_summary(frame, split)
    (RESULTS_DIR / "multihorizon_temporal_split.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return frame, split, raw


def build_feature_matrix(
    frame: pd.DataFrame,
    feature_columns: list[str] | None = None,
) -> pd.DataFrame:
    matrix = pd.get_dummies(
        frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
        columns=CATEGORICAL_FEATURES,
        drop_first=False,
        dtype=float,
    )
    if feature_columns is None:
        return matrix.astype(np.float32)
    for column in feature_columns:
        if column not in matrix:
            matrix[column] = 0.0
    return matrix.reindex(columns=feature_columns, fill_value=0.0).astype(np.float32)


def regression_metrics(actual: Any, predicted: Any) -> dict[str, float]:
    y_true = np.asarray(actual, dtype=float)
    y_pred = np.clip(np.asarray(predicted, dtype=float), 0.0, None)
    return {
        "rmse_ug_m3": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_ug_m3": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias_ug_m3": float(np.mean(y_pred - y_true)),
    }


def prediction_frame(
    frame: pd.DataFrame,
    mask: pd.Series,
    predictions: np.ndarray,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    part = frame.loc[mask].reset_index(drop=True)
    actual = part[TARGET_COLUMNS].to_numpy(dtype=float)
    predicted = np.clip(np.asarray(predictions, dtype=float), 0.0, None)
    if predicted.shape != actual.shape:
        raise ValueError(f"Prediction shape {predicted.shape} khác target shape {actual.shape}.")
    rows = len(part)
    output = pd.DataFrame({
        "model": model_name,
        "split": split_name,
        "city": np.repeat(part["city"].to_numpy(), len(HORIZONS)),
        "source_time": np.repeat(part["datetime"].to_numpy(), len(HORIZONS)),
        "horizon": np.tile(HORIZONS, rows),
        "actual_pm25": actual.reshape(-1),
        "predicted_pm25": predicted.reshape(-1),
    })
    output["target_time"] = pd.to_datetime(output["source_time"]) + pd.to_timedelta(
        output["horizon"], unit="h"
    )
    output["abs_error_ug_m3"] = np.abs(output["actual_pm25"] - output["predicted_pm25"])
    return output


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    horizon_rows = []
    for (model, split_name, horizon), part in predictions.groupby(
        ["model", "split", "horizon"], sort=True
    ):
        horizon_rows.append({
            "model": model,
            "split": split_name,
            "horizon": int(horizon),
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_horizon = pd.DataFrame(horizon_rows)

    city_rows = []
    for (model, split_name, city), part in predictions.groupby(
        ["model", "split", "city"], sort=True
    ):
        city_rows.append({
            "model": model,
            "split": split_name,
            "city": city,
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_city = pd.DataFrame(city_rows)

    summary_rows = []
    for (model, split_name), part in predictions.groupby(["model", "split"], sort=True):
        horizon_part = by_horizon.loc[
            by_horizon["model"].eq(model) & by_horizon["split"].eq(split_name)
        ]
        summary_rows.append({
            "model": model,
            "split": split_name,
            "rows": int(len(part)),
            "mean_horizon_rmse_ug_m3": float(horizon_part["rmse_ug_m3"].mean()),
            "mean_horizon_mae_ug_m3": float(horizon_part["mae_ug_m3"].mean()),
            **{f"global_{key}": value for key, value in regression_metrics(
                part["actual_pm25"], part["predicted_pm25"]
            ).items()},
        })
    return by_horizon, by_city, pd.DataFrame(summary_rows)


def save_evaluation(model_slug: str, predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    by_horizon, by_city, summary = metric_tables(predictions)
    predictions.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_horizon.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_horizon.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_city.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_city.csv",
        index=False,
        encoding="utf-8-sig",
    )
    summary.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return by_horizon, by_city, summary


In [3]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

MODEL_NAME = "LSTM"
MODEL_SLUG = "lstm"
SEQUENCE_LENGTH = 72
BATCH_SIZE = 256
MAX_EPOCHS = 60
LSTM_FEATURES = [
    "pm25", "pm10", "o3", "no2", "temp", "humidity", "wind_speed",
    "pressure", "precip", "hour_sin", "hour_cos", "day_sin", "day_cos",
]


In [4]:
def make_lstm_sequences(
    frame: pd.DataFrame,
    scaled_features: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    windows = []
    targets = []
    endpoint_positions = []
    all_targets = frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)

    for _, city_frame in frame.groupby("city", sort=False):
        positions = city_frame.index.to_numpy(dtype=int)
        values = scaled_features[positions]
        times = city_frame["datetime"].to_numpy(dtype="datetime64[h]")
        if len(city_frame) < SEQUENCE_LENGTH:
            continue
        city_windows = np.lib.stride_tricks.sliding_window_view(
            values, window_shape=SEQUENCE_LENGTH, axis=0
        ).transpose(0, 2, 1)
        ends = np.arange(SEQUENCE_LENGTH - 1, len(city_frame))
        starts = ends - SEQUENCE_LENGTH + 1
        continuous = (times[ends] - times[starts]) == np.timedelta64(SEQUENCE_LENGTH - 1, "h")
        valid_ends = ends[continuous]
        windows.append(city_windows[continuous])
        endpoint_positions.append(positions[valid_ends])
        targets.append(all_targets[positions[valid_ends]])

    return (
        np.concatenate(windows).astype(np.float32),
        np.concatenate(targets).astype(np.float32),
        np.concatenate(endpoint_positions).astype(int),
    )


In [5]:
model_frame, split, raw_data = load_model_frame()
encoded_city = pd.get_dummies(model_frame["city"], prefix="city", dtype=float)
feature_frame = pd.concat([model_frame[LSTM_FEATURES], encoded_city], axis=1)
lstm_feature_columns = feature_frame.columns.tolist()

train_row_mask = split["masks"]["train"].to_numpy()
feature_scaler = StandardScaler()
feature_scaler.fit(feature_frame.loc[train_row_mask])
scaled_features = feature_scaler.transform(feature_frame).astype(np.float32)

sequences, sequence_targets, endpoint_positions = make_lstm_sequences(
    model_frame, scaled_features
)
endpoint_split_masks = {
    name: mask.to_numpy()[endpoint_positions]
    for name, mask in split["masks"].items()
}

target_scaler = StandardScaler()
target_scaler.fit(sequence_targets[endpoint_split_masks["train"]].reshape(-1, 1))
scaled_targets = target_scaler.transform(sequence_targets.reshape(-1, 1)).reshape(
    sequence_targets.shape
).astype(np.float32)

x_train = sequences[endpoint_split_masks["train"]]
y_train = scaled_targets[endpoint_split_masks["train"]]
x_validation = sequences[endpoint_split_masks["validation"]]
y_validation = scaled_targets[endpoint_split_masks["validation"]]
x_test = sequences[endpoint_split_masks["test"]]
y_test = scaled_targets[endpoint_split_masks["test"]]
print(x_train.shape, x_validation.shape, x_test.shape)


(68616, 72, 16) (14679, 72, 16) (14679, 72, 16)


In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, x_train.shape[-1])),
    tf.keras.layers.LSTM(96, return_sequences=True, dropout=0.15),
    tf.keras.layers.LSTM(48, dropout=0.15),
    tf.keras.layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    ),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(len(HORIZONS), activation="linear"),
], name="pm25_multihorizon_lstm")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[
        tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
    ],
)
model.summary()


Model: "pm25_multihorizon_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 72, 96)         │        43,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 48)             │        27,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,928 (296.59 KB)

 Trainable params: 75,928 (296.59 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, min_delta=1e-4
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(CANDIDATES_DIR / "lstm_multihorizon.keras"),
        monitor="val_loss",
        save_best_only=True,
    ),
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=False,
    verbose=1,
)
history_frame = pd.DataFrame(history.history)
history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
history_frame.to_csv(
    RESULTS_DIR / "lstm_multihorizon_learning_curve.csv",
    index=False,
    encoding="utf-8-sig",
)


Epoch 1/60


  1/269 ━━━━━━━━━━━━━━━━━━━━ 41:46 9s/step - loss: 1.5130 - mae: 0.8199 - rmse: 1.2278

  2/269 ━━━━━━━━━━━━━━━━━━━━ 2:23 538ms/step - loss: 1.3345 - mae: 0.7770 - rmse: 1.1502

  3/269 ━━━━━━━━━━━━━━━━━━━━ 2:03 466ms/step - loss: 1.4575 - mae: 0.8169 - rmse: 1.2012

  4/269 ━━━━━━━━━━━━━━━━━━━━ 1:55 436ms/step - loss: 1.5033 - mae: 0.8366 - rmse: 1.2206

...

267/269 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - loss: 0.5778 - mae: 0.5211 - rmse: 0.7512

268/269 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - loss: 0.5770 - mae: 0.5206 - rmse: 0.7506

269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - loss: 0.5762 - mae: 0.5202 - rmse: 0.7500


Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.


269/269 ━━━━━━━━━━━━━━━━━━━━ 94s 349ms/step - loss: 0.3601 - mae: 0.3993 - rmse: 0.5971 - val_loss: 0.7156 - val_mae: 0.5247 - val_rmse: 0.8438 - learning_rate: 5.0000e-04


In [8]:
def inverse_target_scale(values: np.ndarray) -> np.ndarray:
    return target_scaler.inverse_transform(values.reshape(-1, 1)).reshape(values.shape)

validation_predictions = np.clip(
    inverse_target_scale(model.predict(x_validation, batch_size=BATCH_SIZE, verbose=0)),
    0.0,
    None,
)
test_predictions = np.clip(
    inverse_target_scale(model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)),
    0.0,
    None,
)

endpoint_frame = model_frame.iloc[endpoint_positions].copy()
endpoint_frame.index = np.arange(len(endpoint_frame))
endpoint_split = {
    name: pd.Series(mask, index=endpoint_frame.index)
    for name, mask in endpoint_split_masks.items()
}
predictions = pd.concat([
    prediction_frame(
        endpoint_frame, endpoint_split["validation"], validation_predictions,
        MODEL_NAME, "validation"
    ),
    prediction_frame(
        endpoint_frame, endpoint_split["test"], test_predictions,
        MODEL_NAME, "test"
    ),
], ignore_index=True)

joblib.dump(
    {
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "feature_columns": lstm_feature_columns,
        "sequence_length": SEQUENCE_LENGTH,
        "horizons": HORIZONS.tolist(),
    },
    CANDIDATES_DIR / "lstm_multihorizon_preprocessing.joblib",
)
by_horizon, by_city, summary = save_evaluation(MODEL_SLUG, predictions)
display(summary)
display(by_city)


,model,split,rows,mean_horizon_rmse_ug_m3,mean_horizon_mae_ug_m3,global_rmse_ug_m3,global_mae_ug_m3,global_r2,global_bias_ug_m3
0,LSTM,test,352296,15.259917,9.893819,15.336945,9.893819,0.619100,0.075982
1,LSTM,validation,352296,16.901673,10.503458,17.036844,10.503458,0.540826,-2.458232


,model,split,city,rows,rmse_ug_m3,mae_ug_m3,r2,bias_ug_m3
0,LSTM,test,Hà Nội,117432,23.280297,16.440091,0.459872,-5.091752
1,LSTM,test,TP.HCM,117432,10.799167,8.419504,0.260154,4.292298
2,LSTM,test,Đà Nẵng,117432,6.860860,4.821862,0.540253,1.027400
3,LSTM,validation,Hà Nội,117432,23.848374,15.331742,0.357905,-6.456247
4,LSTM,validation,TP.HCM,117432,14.492655,10.113979,0.355242,-1.556157
5,LSTM,validation,Đà Nẵng,117432,9.590631,6.064654,0.474401,0.637707


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
curve_pairs = [
    ("loss", "val_loss", "MSE loss"),
    ("rmse", "val_rmse", "RMSE chuẩn hóa"),
    ("mae", "val_mae", "MAE chuẩn hóa"),
]
for axis, (train_column, val_column, title) in zip(axes, curve_pairs):
    axis.plot(history_frame["epoch"], history_frame[train_column], label="Train")
    axis.plot(history_frame["epoch"], history_frame[val_column], label="Validation")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.2)
axes[0].legend()
plt.tight_layout()
plt.show()


C:\Users\nguyen\AppData\Local\Temp\ipykernel_25936\1675078938.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
test_curve = by_horizon.loc[by_horizon["split"].eq("test")]
axis = test_curve.plot(
    x="horizon", y=["rmse_ug_m3", "mae_ug_m3"], marker="o", figsize=(9, 4.5)
)
axis.set_xlabel("Chân trời dự báo (giờ)")
axis.set_ylabel("Sai số (µg/m³)")
axis.set_title("LSTM: sai số Test theo chân trời")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()


C:\Users\nguyen\AppData\Local\Temp\ipykernel_25936\851945057.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
best_epoch = int(history_frame["val_loss"].idxmin() + 1)
val_summary = summary.loc[summary["split"].eq("validation")].iloc[0]
test_summary = summary.loc[summary["split"].eq("test")].iloc[0]
display(Markdown(
    f"**Nhận xét.** Validation loss nhỏ nhất tại epoch **{best_epoch}**. Validation "
    f"RMSE trung bình theo horizon là **{val_summary['mean_horizon_rmse_ug_m3']:.2f} "
    f"µg/m³**, còn Test RMSE là **{test_summary['mean_horizon_rmse_ug_m3']:.2f} "
    "µg/m³**. Khoảng cách giữa đường Train và Validation cho biết mức overfitting; "
    "Early Stopping giữ lại trọng số tại epoch có Validation loss tốt nhất."
))


**Nhận xét.** Validation loss nhỏ nhất tại epoch **11**. Validation RMSE trung bình theo horizon là **16.90 µg/m³**, còn Test RMSE là **15.26 µg/m³**. Khoảng cách giữa đường Train và Validation cho biết mức overfitting; Early Stopping giữ lại trọng số tại epoch có Validation loss tốt nhất.